# 🛒 Multi-Site Laptop Scraper
### Noon · Amazon · Jumia (Egypt)

### Made By: Eng. Elsayed Bakry
This notebook scrapes laptop listings from **Noon**, **Amazon**, and **Jumia** Egypt.

**How to use:**
1. Run **Cell 1** once to install dependencies.
2. Run **Cell 2** to import libraries.
3. Run **Cell 3** to define the shared `extract_specs` helper.
4. Run **Cell 4 / 5 / 6** for whichever site(s) you want — each is fully independent.
5. Run **Cell 7** to merge all scraped data into one CSV.

> ⚙️ **Settings** — change `MAX_PAGES` (default **10**) and `OUTPUT_FILE` at the top of each scraper cell.  
> 🔇 To run without opening a browser window, uncomment the `--headless=new` line inside `get_driver()`.


## Cell 1 — Install dependencies
Run this **once**. Skip on subsequent runs if packages are already installed.

In [1]:
!pip install -q selenium pandas webdriver-manager


[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\5g\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Cell 2 — Imports
Standard libraries used by every scraper below.

In [2]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

print("✅ Imports ready")

✅ Imports ready


## Cell 3 — Shared helpers
Two reusable functions used by every scraper:

- **`get_driver()`** — launches a Chrome browser with anti-bot settings.  
- **`extract_specs(title)`** — parses RAM, storage, screen size, CPU, and GPU from a product title using regex.


In [3]:
# ─────────────────────────────────────────────────────────────────
# get_driver() — returns a configured Chrome WebDriver instance
# ─────────────────────────────────────────────────────────────────

def get_driver():
    """Launch Chrome with settings that reduce bot-detection."""
    options = Options()

    # Uncomment the next line to run without opening a browser window:
    # options.add_argument("--headless=new")

    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )
    # Mask navigator.webdriver flag
    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"}
    )
    return driver


# ─────────────────────────────────────────────────────────────────
# extract_specs(title) — parses spec fields from a product title
# ─────────────────────────────────────────────────────────────────

def extract_specs(title):
    """
    Extract hardware specs from a laptop product title using regex.

    Returns a dict with keys:
        screen_size, processor, ram, storage, gpu
    Any field not found in the title will be None.
    """
    specs = {
        "screen_size": None,
        "processor":   None,
        "ram":         None,
        "storage":     None,
        "gpu":         None,
    }

    if not title:
        return specs

    # RAM — e.g. "16GB RAM"
    m = re.search(r'(\d+)\s?GB\s?RAM', title, re.I)
    if m:
        specs["ram"] = m.group(1) + "GB"

    # Storage — e.g. "512GB SSD" / "1TB HDD"
    m = re.search(r'(\d+\s?(?:TB|GB)\s?(?:SSD|HDD|NVMe))', title, re.I)
    if m:
        specs["storage"] = m.group(1)

    # Screen size — e.g. '15.6"' / "14 inch"
    m = re.search(r'(\d+(?:\.\d+)?)\s?["\'\- ]?(?:inch|\")', title, re.I)
    if m:
        specs["screen_size"] = m.group(1) + '"'

    # Processor — Intel Core iX / Ryzen X / Apple M
    m = re.search(
        r'(Intel\s?Core\s?i[3579][\-\s\w]*'
        r'|Ryzen\s?[3579][\-\s\w]*'
        r'|Apple\sM\d)',
        title, re.I
    )
    if m:
        specs["processor"] = m.group(1).strip()

    # GPU — RTX / GTX / RX / Intel UHD
    m = re.search(
        r'(RTX\s?\d{3,4}|GTX\s?\d{3,4}|RX\s?\d{3,4}|Intel\s?UHD)',
        title, re.I
    )
    if m:
        specs["gpu"] = m.group(1)

    return specs


print("✅ Helper functions defined")

✅ Helper functions defined


## Cell 4 — Scrape Noon
Scrapes up to **MAX_PAGES** pages from Noon Egypt's laptop category using **URL-based pagination** (`?page=N`).  
A seen-titles guard stops if the site repeats the same products.


In [4]:
# ── Settings ──────────────────────────────────────────────────────
NOON_BASE_URL = "https://www.noon.com/egypt-en/electronics-and-mobiles/computers-and-accessories/computers-new/laptops/"
MAX_PAGES     = 10          # Max pages to scrape (set to None for all pages)
# ──────────────────────────────────────────────────────────────────


def scrape_noon(base_url=NOON_BASE_URL, max_pages=MAX_PAGES):
    """
    Scrape laptop listings from Noon Egypt using URL-based pagination.

    Noon appends ?page=N to the base URL for pages > 1.
    Page 1  → https://…/laptops/
    Page 2  → https://…/laptops/?page=2
    Page N  → https://…/laptops/?page=N

    Parameters
    ----------
    base_url  : str  – Noon laptops category URL (no page param)
    max_pages : int  – Maximum number of pages (None = scrape until end)

    Returns
    -------
    pd.DataFrame with columns:
        name, price, rating, reviews_count, screen_size,
        processor, ram, storage, gpu, url, image, source
    """
    driver = get_driver()
    all_laptops = []
    seen_titles = set()          # duplicate guard per run

    for page in range(1, (max_pages or 9999) + 1):
        # Build page URL — page 1 uses the bare base URL
        page_url = base_url if page == 1 else f"{base_url}?page={page}"

        print(f"\n{'='*40}")
        print(f"  PAGE {page}  →  {page_url}")
        print(f"{'='*40}")

        driver.get(page_url)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-qa="plp-product-box"]'))
            )
        except Exception:
            print("⚠️  Timed-out waiting for products — stopping.")
            break

        time.sleep(2)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        cards = driver.find_elements(By.CSS_SELECTOR, '[data-qa="plp-product-box"]')
        print(f"Found {len(cards)} products")

        if not cards:
            print("⚠️  No products found — stopping.")
            break

        new_on_page = 0
        for idx, card in enumerate(cards):
            try:
                try:
                    title = card.find_element(By.CSS_SELECTOR, '[data-qa="plp-product-box-name"]').text.strip()
                except Exception:
                    title = None

                # Stop if we've already seen this product (page didn't advance)
                if title and title in seen_titles:
                    continue
                if title:
                    seen_titles.add(title)
                    new_on_page += 1

                try:
                    price = card.find_element(By.CSS_SELECTOR, '[data-qa="plp-product-box-price"] strong').text.strip()
                except Exception:
                    price = None

                try:
                    rating = card.find_element(
                        By.CSS_SELECTOR,
                        '.RatingPreviewStar-module-scss-module__zCpaOG__textCtr'
                    ).text.strip()
                except Exception:
                    rating = None

                try:
                    reviews = card.find_element(
                        By.CSS_SELECTOR,
                        '.RatingPreviewStar-module-scss-module__zCpaOG__countCtr span'
                    ).text.strip()
                except Exception:
                    reviews = None

                try:
                    image = card.find_element(By.CSS_SELECTOR, 'img[class*="productImage"]').get_attribute("src")
                except Exception:
                    image = None

                try:
                    product_url = card.find_element(By.CSS_SELECTOR, 'a').get_attribute("href")
                except Exception:
                    product_url = None

                specs = extract_specs(title)

                all_laptops.append({
                    "name":          title,
                    "price":         price,
                    "rating":        rating,
                    "reviews_count": reviews,
                    "screen_size":   specs["screen_size"],
                    "processor":     specs["processor"],
                    "ram":           specs["ram"],
                    "storage":       specs["storage"],
                    "gpu":           specs["gpu"],
                    "url":           product_url,
                    "image":         image,
                    "source":        "Noon",
                })

                short_title = (title[:65] + "...") if title and len(title) > 65 else (title or "Unknown")
                print(f"  {idx+1:>3}. {short_title}")

            except Exception as e:
                print(f"  ⚠️  Product {idx+1} error: {e}")

        # If zero new products came in, the site didn't advance → stop
        if new_on_page == 0:
            print("\n🏁 No new products on this page — scraping complete.")
            break

    driver.quit()

    df = pd.DataFrame(all_laptops)
    print(f"\n✅ Noon scraping done — {len(df)} products collected")
    return df


# ── Run ───────────────────────────────────────────────────────────
noon_df = scrape_noon()
noon_df.head()



  PAGE 1  →  https://www.noon.com/egypt-en/electronics-and-mobiles/computers-and-accessories/computers-new/laptops/
Found 50 products
    1. Apple MacBook Air MGN63 With 13 Inch Display, M1 Chip 8-Core CPU ...
    2. Apple MacBook Pro MDE34 | 14 Inch Display | Apple M5 Chip | 10-Co...
    3. Apple New 2024 MacBook Air MRXW3 13-inch Display, Apple M3 Chip 8...
    4. Apple MacBook Pro MDE04 | 14 Inch Display | Apple M5 Chip | 10-Co...
    5. Apple MacBook Pro MX2H3 Liquid XDR Retina 14-Inch Display, M4 Pro...
    6. Apple MacBook Air MW0W3 | 13 Inch Display | Apple M4 Chip | 10-Co...
    7. Lenovo LAPTOP LENOVO LOQ 15AHP10 AMD RYZEN 7 250 RAM 16G DDR5 560...
    8. ASUS FX608JMR-RV017W ASUS TUF Gaming F16 Laptop with 16 inch FHD ...
    9. Apple New 2026 MacBook Air MDHE4 | 13 Inch Display | Apple M5 Air...
   10. Apple MacBook Air MC6T4 | 13 Inch Display | Apple M4 Chip | 10-Co...
   11. Apple New 2026 MacBook Air MDVH4 | 15 Inch Display | Apple M5 Air...
   12. HP Victus 15‑fa1035NE 

,name,price,rating,reviews_count,screen_size,processor,ram,storage,gpu,url,image,source
0,"Apple MacBook Air MGN63 With 13 Inch Display, ...","39,333",4.6,5.6K,"13""",None,8GB,256GB SSD,None,https://www.noon.com/egypt-en/macbook-air-mgn6...,https://f.nooncdn.com/p/pzsku/Z073CC3975B50405...,Noon
1,Apple MacBook Pro MDE34 | 14 Inch Display | Ap...,"114,944",4.5,666,"14""",Apple M5,24GB,1TB SSD,None,https://www.noon.com/egypt-en/macbook-pro-mde3...,https://f.nooncdn.com/p/pzsku/ZB86807EF11E0C21...,Noon
2,Apple New 2024 MacBook Air MRXW3 13-inch Displ...,"51,999",4.6,724,"13""",Apple M3,8GB,512GB SSD,Intel UHD,https://www.noon.com/egypt-en/new-2024-macbook...,https://f.nooncdn.com/p/pzsku/ZAAE18FC6CBA31C4...,Noon
3,Apple MacBook Pro MDE04 | 14 Inch Display | Ap...,"97,433",4.5,666,"14""",Apple M5,16GB,512GB SSD,None,https://www.noon.com/egypt-en/macbook-pro-mde0...,https://f.nooncdn.com/p/pzsku/Z2276A3E16D90EAC...,Noon
4,Apple MacBook Pro MX2H3 Liquid XDR Retina 14-I...,"119,508",4.5,666,"14""",None,24GB,512GB SSD,None,https://www.noon.com/egypt-en/macbook-pro-mx2h...,https://f.nooncdn.com/p/pzsku/ZDC348E47B0C87C4...,Noon


## Cell 5 — Scrape Amazon Egypt
Scrapes up to **MAX_PAGES** pages from the Amazon Egypt laptop search results using **URL-based pagination** (`&page=N`).  
A seen-ASINs guard stops if Amazon repeats results.


In [5]:
# ── Settings ──────────────────────────────────────────────────────
AMAZON_BASE_URL = "https://www.amazon.eg/s?k=laptops&i=electronics"
MAX_PAGES       = 10          # Max pages to scrape (set to None for all pages)
# ──────────────────────────────────────────────────────────────────


def scrape_amazon(base_url=AMAZON_BASE_URL, max_pages=MAX_PAGES):
    """
    Scrape laptop listings from Amazon Egypt using URL-based pagination.

    Amazon appends &page=N to the search URL.
    Page 1  → https://…?k=laptops&i=electronics
    Page 2  → https://…?k=laptops&i=electronics&page=2
    Page N  → https://…?k=laptops&i=electronics&page=N

    Parameters
    ----------
    base_url  : str  – Amazon search URL (no page param)
    max_pages : int  – Maximum number of pages (None = scrape until end)

    Returns
    -------
    pd.DataFrame with columns:
        name, price, rating, reviews_count, screen_size,
        processor, ram, storage, gpu, url, image, source
    """
    driver = get_driver()
    all_laptops = []
    seen_asins  = set()          # duplicate guard per run

    for page in range(1, (max_pages or 9999) + 1):
        page_url = base_url if page == 1 else f"{base_url}&page={page}"

        print(f"\n{'='*40}")
        print(f"  PAGE {page}  →  {page_url}")
        print(f"{'='*40}")

        driver.get(page_url)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]'))
            )
        except Exception:
            print("⚠️  Timed-out waiting for products — stopping.")
            break

        time.sleep(3)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        products = driver.find_elements(By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
        print(f"Found {len(products)} products")

        if not products:
            print("⚠️  No products found — stopping.")
            break

        new_on_page = 0
        for idx, product in enumerate(products):
            try:
                asin = product.get_attribute("data-asin")
                if not asin:          # sponsored card with no ASIN
                    continue
                if asin in seen_asins:
                    continue
                seen_asins.add(asin)
                new_on_page += 1

                try:
                    title = product.find_element(By.CSS_SELECTOR, 'h2 span').text.strip()
                except Exception:
                    title = None

                try:
                    whole    = product.find_element(By.CSS_SELECTOR, '.a-price-whole').text.strip()
                    fraction = product.find_element(By.CSS_SELECTOR, '.a-price-fraction').text.strip()
                    price    = whole + "." + fraction
                except Exception:
                    price = None

                try:
                    raw_rating = product.find_element(By.CSS_SELECTOR, '.a-icon-alt').get_attribute("innerHTML")
                    rating = raw_rating.split(" ")[0]
                except Exception:
                    rating = None

                try:
                    reviews = product.find_element(By.CSS_SELECTOR, '.a-size-base.s-underline-text').text.strip()
                except Exception:
                    reviews = None

                try:
                    image = product.find_element(By.CSS_SELECTOR, 'img.s-image').get_attribute("src")
                except Exception:
                    image = None

                try:
                    product_url = product.find_element(By.CSS_SELECTOR, 'a.a-link-normal.s-line-clamp-4').get_attribute("href")
                except Exception:
                    product_url = None

                specs = extract_specs(title)

                all_laptops.append({
                    "name":          title,
                    "price":         price,
                    "rating":        rating,
                    "reviews_count": reviews,
                    "screen_size":   specs["screen_size"],
                    "processor":     specs["processor"],
                    "ram":           specs["ram"],
                    "storage":       specs["storage"],
                    "gpu":           specs["gpu"],
                    "url":           product_url,
                    "image":         image,
                    "source":        "Amazon",
                })

                short_title = (title[:65] + "...") if title and len(title) > 65 else (title or "Unknown")
                print(f"  {idx+1:>3}. {short_title}")

            except Exception as e:
                print(f"  ⚠️  Product {idx+1} error: {e}")

        # No new ASINs → Amazon didn't advance (last page repeated)
        if new_on_page == 0:
            print("\n🏁 No new products on this page — scraping complete.")
            break

    driver.quit()

    df = pd.DataFrame(all_laptops)
    print(f"\n✅ Amazon scraping done — {len(df)} products collected")
    return df


# ── Run ───────────────────────────────────────────────────────────
amazon_df = scrape_amazon()
amazon_df.head()



  PAGE 1  →  https://www.amazon.eg/s?k=laptops&i=electronics
Found 27 products
    1. Lenovo LOQ 15IRX9 Gaming Laptop - 13th Intel Core i7-13650HX 14 C...
    2. Lenovo LOQ 15IRX10 Gaming Laptop, Intel Core i7-14700HX, 16GB RAM...
    3. MSI Katana 15 HX B14WFK (NVIDIA® GeForce RTX™ 5060 Laptop GPU, GD...
    4. HP Laptop 14-em0025ne: Ryzen 5-7520U, 8GB Ram, 512GB-SSD, AMD Rad...
    5. ASUS Vivobook 14 [X1405VA-H005W] i5-13420H / 8GB DDR4 / 512GB SSD...
    6. ASUS Vivobook 16 [X1605VA-ACH007W] Core™ i7-13620H /16GB DDR5 /51...
    7. Lenovo Ideapad Slim 3 15IRH10 Laptop Intel Core i5 13420H, 16GB R...
    8. HP OmniBook 3 AI PC15-fn0001ne: Ryzen AI 5 330, 16GB, 512GB-SSD, ...
    9. ASUS TUF F16 [FX607VJB-RL165W] Intel Core 5-210H /16GB DDR5 /512G...
   10. Dell Latitude 5310 2 in 1 Laptop Touch Screen 13.3" FHD (1920 x 1...
   11. Dell Latitude 5300 Touchscreen Laptop with Backlit Keyboard, 13.3...
   12. Dell Latitude 5310 2-in-1 Touchscreen Business Laptop, 13.3" FHD ...
   13. H

,name,price,rating,reviews_count,screen_size,processor,ram,storage,gpu,url,image,source
0,Lenovo LOQ 15IRX9 Gaming Laptop - 13th Intel C...,"53,999.00",4.0,None,"15.6""",Intel Core i7-13650HX 14 Cores,None,512GB SSD,RTX 4050,https://www.amazon.eg/-/en/sspa/click?ie=UTF8&...,https://m.media-amazon.com/images/I/61wFLokn9R...,Amazon
1,"Lenovo LOQ 15IRX10 Gaming Laptop, Intel Core i...","68,999.00",None,None,"15.6""",Intel Core i7-14700HX,16GB,512GB SSD,RTX 5060,https://www.amazon.eg/-/en/sspa/click?ie=UTF8&...,https://m.media-amazon.com/images/I/71U2lhrNVM...,Amazon
2,MSI Katana 15 HX B14WFK (NVIDIA® GeForce RTX™ ...,"67,499.00",None,None,None,None,None,None,None,https://www.amazon.eg/-/en/sspa/click?ie=UTF8&...,https://m.media-amazon.com/images/I/71YTBqh1xe...,Amazon
3,"HP Laptop 14-em0025ne: Ryzen 5-7520U, 8GB Ram,...","19,999.00",4.2,None,"14""",Ryzen 5-7520U,8GB,None,None,https://www.amazon.eg/-/en/HP-Laptop-14-em0025...,https://m.media-amazon.com/images/I/51kHkSKx3q...,Amazon
4,ASUS Vivobook 14 [X1405VA-H005W] i5-13420H / 8...,"26,999.00",None,None,"14.0""",None,None,512GB SSD,None,https://www.amazon.eg/-/en/ASUS-Vivobook-X1405...,https://m.media-amazon.com/images/I/71u79SvGtA...,Amazon


## Cell 6 — Scrape Jumia Egypt
Scrapes up to **MAX_PAGES** pages from Jumia Egypt's laptop section using **URL-based pagination** (`?page=N#catalog-listing`).  
Note: Jumia does not display ratings/review counts on listing pages.


In [6]:
# ── Settings ──────────────────────────────────────────────────────
JUMIA_BASE_URL = "https://www.jumia.com.eg/laptops/"
MAX_PAGES      = 10          # Max pages to scrape (set to None for all pages)
# ──────────────────────────────────────────────────────────────────


def scrape_jumia(base_url=JUMIA_BASE_URL, max_pages=MAX_PAGES):
    """
    Scrape laptop listings from Jumia Egypt using URL-based pagination.

    Jumia appends ?page=N#catalog-listing to the base URL.
    Page 1  → https://…/laptops/
    Page 2  → https://…/laptops/?page=2#catalog-listing
    Page N  → https://…/laptops/?page=N#catalog-listing

    Parameters
    ----------
    base_url  : str  – Jumia laptops category URL (no page param)
    max_pages : int  – Maximum number of pages (None = scrape until end)

    Returns
    -------
    pd.DataFrame with columns:
        name, price, rating, reviews_count, screen_size,
        processor, ram, storage, gpu, url, image, source

    Note: Jumia listing pages do not expose rating / reviews_count;
          those fields will be None.
    """
    driver = get_driver()
    all_laptops = []
    seen_urls   = set()          # duplicate guard per run

    for page in range(1, (max_pages or 9999) + 1):
        page_url = base_url if page == 1 else f"{base_url}?page={page}#catalog-listing"

        print(f"\n{'='*40}")
        print(f"  PAGE {page}  →  {page_url}")
        print(f"{'='*40}")

        driver.get(page_url)

        try:
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "article.prd"))
            )
        except Exception:
            print("⚠️  Timed-out waiting for products — stopping.")
            break

        time.sleep(3)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        products = driver.find_elements(By.CSS_SELECTOR, "article.prd")
        print(f"Found {len(products)} products")

        if not products:
            print("⚠️  No products found — stopping.")
            break

        new_on_page = 0
        for idx, product in enumerate(products):
            try:
                try:
                    title = product.find_element(By.CSS_SELECTOR, "h3.name").text.strip()
                except Exception:
                    title = None

                try:
                    product_url = product.find_element(By.CSS_SELECTOR, "a.core").get_attribute("href")
                except Exception:
                    product_url = None

                # Deduplicate by URL (or title if URL missing)
                dedup_key = product_url or title
                if dedup_key and dedup_key in seen_urls:
                    continue
                if dedup_key:
                    seen_urls.add(dedup_key)
                    new_on_page += 1

                try:
                    price = product.find_element(By.CSS_SELECTOR, "div.prc").text.strip()
                except Exception:
                    price = None

                try:
                    image = product.find_element(By.CSS_SELECTOR, "img.img").get_attribute("src")
                except Exception:
                    image = None

                specs = extract_specs(title)

                all_laptops.append({
                    "name":          title,
                    "price":         price,
                    "rating":        None,   # Not available on listing page
                    "reviews_count": None,   # Not available on listing page
                    "screen_size":   specs["screen_size"],
                    "processor":     specs["processor"],
                    "ram":           specs["ram"],
                    "storage":       specs["storage"],
                    "gpu":           specs["gpu"],
                    "url":           product_url,
                    "image":         image,
                    "source":        "Jumia",
                })

                short_title = (title[:65] + "...") if title and len(title) > 65 else (title or "Unknown")
                print(f"  {idx+1:>3}. {short_title}")

            except Exception as e:
                print(f"  ⚠️  Product {idx+1} error: {e}")

        # No new products → Jumia didn't advance (last page repeated)
        if new_on_page == 0:
            print("\n🏁 No new products on this page — scraping complete.")
            break

    driver.quit()

    df = pd.DataFrame(all_laptops)
    print(f"\n✅ Jumia scraping done — {len(df)} products collected")
    return df


# ── Run ───────────────────────────────────────────────────────────
jumia_df = scrape_jumia()
jumia_df.head()



  PAGE 1  →  https://www.jumia.com.eg/laptops/
Found 40 products
    1. HP 14-em0025ne Laptop AMD Ryzen™ 5 7520U, 8GB Ram, 512GB SSD, AMD...
    2. HP Victus 15-fb3015ne Laptop - AMD Ryzen™ 7-7445HS - 16GB - 512GB...
    3. Lenovo Legion Go
    4. Asus TUF Gaming A15 FA506NCR-HN007W - AMD Ryzen 7-7435HS , 8GB RA...
    5. Asus LapTop TUF Gaming A15 FA506NCR-HN007W - 15.6 Inch - AMD Ryze...
    6. Asus Vivobook 16 Laptop, Intel® Core™ i7-13620H, 16GB Ram, 512GB ...
    7. Lenovo 83Dv008Qed-Loq 15Irx9 Intel® Core™ I7-13650Hx , Ram 16Gb ,...
    8. Lenovo Loq 15IAX9 Laptop, Intel® Core i5-12450HX, 12GB Ram, 512GB...
    9. Asus LapTop Vivobook S 15-S5507QA-MA037W - 15.6 3K OLED - Snapdra...
   10. MSI Thin A15 B7V Gaming Laptop, AMD Ryzen™ 7 7735HS, 16GB Ram, 51...
   11. Asus TUF F16 Gaming Laptop, Intel® Core™ 5 210H, 8GB Ram, 512 SSD...
   12. Asus TUF A16 Gaming Laptop, AMD Ryzen™ 7 260, 16GB Ram, 512GB SSD...
   13. HP Chromebook 14-inch HD Laptop, Intel Celeron N4000, 4 GB RAM, 3..

,name,price,rating,reviews_count,screen_size,processor,ram,storage,gpu,url,image,source
0,"HP 14-em0025ne Laptop AMD Ryzen™ 5 7520U, 8GB ...","EGP 24,869.00",None,None,None,None,8GB,512GB SSD,None,https://www.jumia.com.eg/hp-14-em0025ne-laptop...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,Jumia
1,HP Victus 15-fb3015ne Laptop - AMD Ryzen™ 7-74...,"EGP 46,689.00",None,None,None,None,None,512GB SSD,None,https://www.jumia.com.eg/hp-victus-15-fb3015ne...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,Jumia
2,Lenovo Legion Go,"EGP 42,559.00",None,None,None,None,None,None,None,https://www.jumia.com.eg/lenovo-legion-go-1077...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,Jumia
3,Asus TUF Gaming A15 FA506NCR-HN007W - AMD Ryze...,"EGP 40,599.00",None,None,None,Ryzen 7-7435HS,8GB,512GB SSD,RTX 3050,https://www.jumia.com.eg/asus-tuf-gaming-a15-f...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,Jumia
4,Asus LapTop TUF Gaming A15 FA506NCR-HN007W - 1...,"EGP 37,999.00",None,None,"15.6""",Ryzen 7 - 8GB,None,512GB SSD,RTX3050,https://www.jumia.com.eg/asus-laptop-tuf-gamin...,https://eg.jumia.is/unsafe/fit-in/300x300/filt...,Jumia


## Cell 7 — Merge all results & save to CSV

Combines whichever DataFrames exist (`noon_df`, `amazon_df`, `jumia_df`) into a  
single file called **`all_laptops.csv`**.

> If you only ran one or two scrapers, the missing DataFrames are simply skipped.


In [8]:
# ── Collect whichever DataFrames were created ────────────────────
available = {}
for name, var in [("Noon", "noon_df"), ("Amazon", "amazon_df"), ("Jumia", "jumia_df")]:
    try:
        df_check = eval(var)
        if not df_check.empty:
            available[name] = df_check
            print(f"  ✅ {name:<8} — {len(df_check):>4} products")
        else:
            print(f"  ⚠️  {name:<8} — DataFrame is empty, skipping")
    except NameError:
        print(f"  ℹ️  {name:<8} — not scraped yet, skipping")

if not available:
    print("\n❌ No data to merge. Run at least one scraper first.")
else:
    OUTPUT_FILE = "all_laptop.csv"

    combined = pd.concat(available.values(), ignore_index=True)

    combined.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print(f"\n{'─'*40}")
    print(f"  Total products : {len(combined)}")
    print(f"  Saved to       : {OUTPUT_FILE}")
    print(f"{'─'*40}")

    combined.head(10)

  ✅ Noon     —  500 products
  ✅ Amazon   —  193 products
  ✅ Jumia    —  370 products

────────────────────────────────────────
  Total products : 1063
  Saved to       : all_laptop.csv
────────────────────────────────────────
